## My Agent Loop

After following the extra activity of creating the agent loop it is time to create it by myself.

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [4]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [5]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [6]:
openai = OpenAI()

In [7]:
# Some lists!

todos = []
completed = []

In [16]:
def get_todo_report() -> str:
    result = ""
    for index,todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: [yellow]{todo}[/yellow]\n"
    show(result)
    return result

In [14]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False]*len(descriptions)) 
    return get_todo_report()

In [17]:
todos, completed = [], []

create_todos(["Play the guitar", "Invest in the stock market", "Travel from Barcelona to Mumbai"])

Todo #1: Play the guitar
Todo #2: Invest in the stock market
Todo #3: Travel from Barcelona to Mumbai

'Todo #1: [yellow]Play the guitar[/yellow]\nTodo #2: [yellow]Invest in the stock market[/yellow]\nTodo #3: [yellow]Travel from Barcelona to Mumbai[/yellow]\n'

In [20]:
def mark_complete(index:int, completion_notes:str) -> str:
    if 1 <= index <= len(todos):
        completed[index-1] = True
    else:
        return "No todo at this index."
    show(completion_notes)
    
    return get_todo_report()

In [21]:
mark_complete(1,"Learned how to play it")

Learned how to play it

Todo #1: Play the guitar
Todo #2: Invest in the stock market
Todo #3: Travel from Barcelona to Mumbai

'Todo #1: [green][strike]Play the guitar[/strike][/green]\nTodo #2: [yellow]Invest in the stock market[/yellow]\nTodo #3: [yellow]Travel from Barcelona to Mumbai[/yellow]\n'

In [22]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [29]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "index": {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            "completion_notes":{
                'description': 'Notes about how you completed the todo in rich console markup',
                'title':'Completion notes',
                'type':'string'
            }
            },
        "required": ["index","completion_notes"],
        "additionalProperties": False
    }
}

In [30]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [35]:
def handle_tool_calls(tool_calls):
    results = []

    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})

    return results

In [36]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [37]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
I would like to learn how to do value investing in order to evaluate stocks.
Then I would invest in the best companies I can find.
How can I do it?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [38]:
todos, completed = [], []
loop(messages)

Todo #1: Outline a practical value-investing workflow from idea generation to decision-making and monitoring
Todo #2: Define core concepts (intrinsic value, margin of safety, moats, owner earnings, capital allocation) and 
key metrics
Todo #3: Provide a step-by-step checklist for analyzing a company’s business, financials, valuation, and risks
Todo #4: Give simple valuation methods (DCF, multiples, earnings power) with guidance on inputs and reasonable 
estimates
Todo #5: Describe portfolio construction, position sizing, buy/sell discipline, and common pitfalls
Todo #6: List learning resources, tools, and a suggested 30/60/90-day study plan

Core concepts + key metrics
- Intrinsic value: your estimate of what the whole business is worth today based on the cash it can generate for 
owners.
- Margin of safety: buy only when price is meaningfully below conservative intrinsic value (often 20–40%+).
- Circle of competence: stick to businesses you can explain (how they make money, key drivers, why customers stay).
- Economic moat: durable advantage (switching costs, network effects, cost advantage, brand, regulation, scale).
- Owner earnings / free cash flow (FCF): cash available to owners after maintaining the business.
  - Common proxy: FCF = Operating Cash Flow − Capex (sometimes adjust for stock comp, working capital, cyclical 
capex).
- Capital allocation: how management reinvests or returns cash (reinvest at high returns, acquisitions, buybacks, 
dividends, debt paydown).

Key metrics (use trend + peers, not one-year snapshots):
- Profitability: gross/operating/net margins; stability over time.
- Returns: ROIC (best), ROE/ROA (watch leverage). Prefer ROIC > WACC and persistent.
- Growth quality: revenue/FCF per share growth; dilution vs buybacks.
- Financial strength: net debt/EBITDA, interest coverage, current ratio (context-dependent).
- Cash conversion: FCF margin, FCF vs net income, working-capital needs.
- Valuation: EV/EBIT, EV/FCF, P/E (with normalized earnings), P/B (mainly for financials), shareholder yield 
(buybacks+dividends).

Todo #1: Outline a practical value-investing workflow from idea generation to decision-making and monitoring
Todo #2: Define core concepts (intrinsic value, margin of safety, moats, owner earnings, capital allocation) and 
key metrics
Todo #3: Provide a step-by-step checklist for analyzing a company’s business, financials, valuation, and risks
Todo #4: Give simple valuation methods (DCF, multiples, earnings power) with guidance on inputs and reasonable 
estimates
Todo #5: Describe portfolio construction, position sizing, buy/sell discipline, and common pitfalls
Todo #6: List learning resources, tools, and a suggested 30/60/90-day study plan

Practical workflow
1) Screen / source ideas: simple screens (FCF positive, high ROIC, low leverage, insider ownership), 52-week lows, 
spin-offs, special situations, quality compounders temporarily down.
2) Understand the business: how it makes money, unit economics, customers, competition, cyclicality.
3) Read filings: 10-K/annual report first; then 10-Q; then investor presentations (last).
4) Assess quality: moat + management + balance sheet + reinvestment runway.
5) Value it conservatively: use 2–3 methods; triangulate; require margin of safety.
6) Decide: write a 1–2 page thesis with base/bull/bear, key risks, and what would prove you wrong.
7) Buy: size position based on conviction and downside; prefer limit orders; be patient.
8) Monitor: track thesis KPIs each quarter; watch capital allocation and competitive position; ignore noise.
9) Exit: when price exceeds conservative value, thesis breaks, or you find a clearly better risk-adjusted idea.

Todo #1: Outline a practical value-investing workflow from idea generation to decision-making and monitoring
Todo #2: Define core concepts (intrinsic value, margin of safety, moats, owner earnings, capital allocation) and 
key metrics
Todo #3: Provide a step-by-step checklist for analyzing a company’s business, financials, valuation, and risks
Todo #4: Give simple valuation methods (DCF, multiples, earnings power) with guidance on inputs and reasonable 
estimates
Todo #5: Describe portfolio construction, position sizing, buy/sell discipline, and common pitfalls
Todo #6: List learning resources, tools, and a suggested 30/60/90-day study plan

...

Portfolio + discipline + pitfalls
- Diversification: 10–20 positions is common for individual value investors; fewer requires higher certainty.
- Position sizing: start 2–5% per position; scale to 5–10% only when downside is limited and thesis is strong.
- Cash: holding cash is acceptable when you can’t find bargains.
- Buy discipline: predefine intrinsic value range; buy only below your “max buy price” (value × (1 − margin of 
safety)).
- Sell discipline: sell/trim if (a) price far exceeds value, (b) thesis broken, (c) better opportunity, (d) balance
sheet risk rises.

Common pitfalls
- Overconfidence in forecasts; ignoring cyclicality.
- Value traps: cheap for a reason (shrinking industry, eroding moat, leverage).
- Using P/E on peak earnings or ignoring dilution.
- Anchoring to past prices; failing to update thesis with new facts.
- Overtrading based on headlines instead of fundamentals.

Todo #1: Outline a practical value-investing workflow from idea generation to decision-making and monitoring
Todo #2: Define core concepts (intrinsic value, margin of safety, moats, owner earnings, capital allocation) and 
key metrics
Todo #3: Provide a step-by-step checklist for analyzing a company’s business, financials, valuation, and risks
Todo #4: Give simple valuation methods (DCF, multiples, earnings power) with guidance on inputs and reasonable 
estimates
Todo #5: Describe portfolio construction, position sizing, buy/sell discipline, and common pitfalls
Todo #6: List learning resources, tools, and a suggested 30/60/90-day study plan

Resources + tools + 30/60/90 plan
Books
- Benjamin Graham: The Intelligent Investor (mindset + margin of safety)
- Buffett letters (free online) (business quality + capital allocation)
- Pat Dorsey: The Little Book That Builds Wealth (moats)
- Greenwald: Value Investing (EPV + competitive advantage)
- Damodaran (NYU, free) for valuation lectures/spreadsheets

Primary sources/tools
- SEC EDGAR (10-K/10-Q), company annual reports
- Quick data: Koyfin, Finviz, Morningstar, Seeking Alpha transcripts (use transcripts carefully)
- Spreadsheet for per-share, 10-year trends, and valuation scenarios

30/60/90 day plan
- Days 1–30: read 10-Ks of 3 simple businesses; write 1-page summaries; learn accounting basics (income/BS/CF 
link).
- Days 31–60: build a one-tab model: normalized EBIT/FCF + EPV; do 5 companies; practice margin-of-safety pricing.
- Days 61–90: add a simple DCF + scenario table; create an “investment memo” template; paper-trade decisions and 
compare outcomes to thesis, not price moves.

Todo #1: Outline a practical value-investing workflow from idea generation to decision-making and monitoring
Todo #2: Define core concepts (intrinsic value, margin of safety, moats, owner earnings, capital allocation) and 
key metrics
Todo #3: Provide a step-by-step checklist for analyzing a company’s business, financials, valuation, and risks
Todo #4: Give simple valuation methods (DCF, multiples, earnings power) with guidance on inputs and reasonable 
estimates
Todo #5: Describe portfolio construction, position sizing, buy/sell discipline, and common pitfalls
Todo #6: List learning resources, tools, and a suggested 30/60/90-day study plan

How to do value investing (a practical way to evaluate stocks)

1) The value-investing workflow (repeatable process)

1. Find candidates: screens (profitable, FCF-positive, reasonable debt), “good companies having a bad year,” 
spin-offs, unpopular sectors, simple businesses you understand.
2. Understand the business (before modeling): how it makes money, why customers stay, what could kill it.
3. Read primary documents: start with the latest 10‑K/annual report (business, risks, accounting), then 10‑Qs.
4. Judge quality: moat + management + balance sheet + reinvestment opportunity.
5. Estimate intrinsic value using 2–3 conservative methods.
6. Require a margin of safety: only buy when price is materially below your conservative value.
7. Write a short investment memo: thesis, assumptions, bear case, and what would prove you wrong.
8. Buy and monitor: track thesis KPIs quarterly (not headlines).
9. Sell/trim: when it’s clearly above value, the thesis breaks, or a better opportunity appears.


2) Core concepts + what to look for
Intrinsic value: what the whole business is worth based on cash it can generate for owners.  
Margin of safety: a discount (often ~20–40%+) to protect you from being wrong.  
Moat: durable advantage (switching costs, network effects, cost edge, brand, regulation, scale).  
Owner earnings / FCF: cash that can be taken out without harming the business (often proxied by Operating Cash Flow
− Capex, adjusted when needed).  
Capital allocation: whether management reinvests, buys back shares, pays dividends, acquires companies, or repays 
debt—and whether they do so at good prices.

Useful metrics (look at 5–10 year trends):
- ROIC (prefer high and stable), margins stability
- FCF consistency and FCF vs net income
- Balance sheet: net debt/EBITDA, interest coverage, liquidity
- Per-share outcomes: share count (dilution vs buybacks), FCF/share growth

3) Company analysis checklist (use this every time)
A) Business
- What’s the product, who pays, and why do they choose it?
- What drives revenue (price/volume/retention)? What drives costs?
- Cyclical or resilient? Sensitive to rates/commodities/FX?

B) Financials
- Any one-time “adjustments” that keep repeating?
- Are margins and returns deteriorating?
- Is cash flow real (or propped up by working-capital tricks)?
- Does the company need heavy capex just to stand still?

C) Management
- Incentives: are they rewarded for ROIC/FCF (good) or just EPS (often gameable)?
- Buybacks: are they buying back shares when cheap (good) or when expensive (bad)?

D) Risks
- Customer concentration, regulation, disruption, leverage/refinancing
- Accounting red flags (aggressive revenue recognition, receivables rising faster than sales)

E) Thesis clarity
- Why mispriced now? What closes the gap? What evidence would disprove you?

4) Valuation methods you can actually use (and reasonable estimates)
Method 1 — Earnings Power Value (EPV) (great for stable businesses)
- Normalize EBIT across a cycle → convert to after-tax operating profit (NOPAT)
- Approximate owner earnings ≈ NOPAT + D&A − maintenance capex (and adjust for working capital if needed)
- Value ≈ Owner earnings ÷ discount rate  
  Reasonable discount rate estimate: ~9–12% for equities; higher for riskier/cyclical names.

Method 2 — Simple DCF (use conservative inputs)
- Forecast 5–10 years of FCF with simple drivers (growth, margins, reinvestment)
- Terminal value:
  - Perpetual growth: ~2–3% (avoid aggressive numbers), or
  - Exit multiple on year-10 EBIT/FCF (keep it conservative)
- Discount back at ~9–12%
- Run base/bull/bear; treat the conservative case as your anchor

Method 3 — Multiples (triangulate)
- Use EV/EBIT or EV/FCF on normalized earnings
- Compare to: its own history + peers + what quality justifies
- Adjust for leverage and cyclicality (cheap cyclicals at peak earnings are a common trap)

Your “max buy price” = conservative intrinsic value × (1 − margin of safety)

5) Portfolio construction + discipline
